In [12]:
# !pip install sentence-transformers scikit-learn pandas

In [13]:
import os
import pickle
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
df = pd.read_csv("../data/processed/train.csv")

df = df.dropna(subset=["question", "answer"])
df = df.drop_duplicates(subset=["question"])

print("Veri sayısı:", len(df))
df.head()

Veri sayısı: 524


,question,answer
0,what does memantine look like,"Color - ORANGE, Shape - CAPSULE (biconvex), Sc..."
1,how does a 20 mcg bedford norton transdermal p...,25 mcg fentanyl/hr = 40 mg oral / 20 mg IV oxy...
2,what mg norco comes in,... NORCO® 5/325 ... NORCO® 7.5/325 ... NORCO®...
3,vyvanse 10 what is all in this pill is it safe,The following adverse reactions are discussed ...
4,dtap/tdap/td vaccines how often is this due,"Routine Vaccination of Infants and Children, A..."


In [15]:
model = SentenceTransformer("all-mpnet-base-v2")

c:\Models\LLM-From-Scratch\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Laptop Dunyası\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3372.73it/s]


In [16]:
questions = df["question"].tolist()
answers = df["answer"].tolist()

question_embeddings = model.encode(
    questions,
    convert_to_tensor=False,
    show_progress_bar=True
)

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches: 100%|██████████| 17/17 [00:12<00:00,  1.39it/s]


In [17]:
def answer_question(user_question, top_k=3):
    user_embedding = model.encode([user_question])

    similarities = cosine_similarity(
        user_embedding,
        question_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    for idx in top_indices:
        results.append({
            "matched_question": questions[idx],
            "answer": answers[idx],
            "score": similarities[idx]
        })

    return results

In [18]:
results = answer_question("Can I take ibuprofen with food?")

for i, result in enumerate(results, 1):
    print(f"\nSonuç {i}")
    print("Benzerlik:", round(result["score"], 4))
    print("Eşleşen soru:", result["matched_question"])
    print("Cevap:", result["answer"][:700])


Sonuç 1
Benzerlik: 0.5443
Eşleşen soru: can i eat after taking rapaflo?
Cevap: The recommended dose is 8 mg orally once daily with a meal.

Sonuç 2
Benzerlik: 0.5336
Eşleşen soru: do i take a statin with or without food ?
Cevap: Read the label on the bottle carefully. Some brands should be taken with food. Others may be taken with, or without food.

Sonuç 3
Benzerlik: 0.5306
Eşleşen soru: what foods should a person taking warfarin eat
Cevap: Eat a normal, healthy diet. Some foods and beverages, particularly those that contain vitamin K, can affect how warfarin works for you. Ask your doctor or pharmacist for a list of foods that contain vitamin K. Eat consistent amounts of vitamin K-containing food on a week-to-week basis. Do not eat large amounts of leafy, green vegetables or certain vegetable oils that contain large amounts of vitamin K. Be sure to talk to your doctor before you make any changes in your diet. Talk to your doctor about eating grapefruit and drinking grapefruit juice 

In [21]:
os.makedirs("../models/distilbert_retriever", exist_ok=True)

model.save("../models/distilbert_retriever/model")

with open("../models/distilbert_retriever/questions.pkl", "wb") as f:
    pickle.dump(questions, f)

with open("../models/distilbert_retriever/answers.pkl", "wb") as f:
    pickle.dump(answers, f)

with open("../models/distilbert_retriever/question_embeddings.pkl", "wb") as f:
    pickle.dump(question_embeddings, f)

print("DistilBERT retriever dosyaları kaydedildi.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]

DistilBERT retriever dosyaları kaydedildi.
